# CALOR Y MAYORES EN MADRID

## Limpieza de datos: centros de mayores

**Fuente:** Registro de centros de atención social, Comunidad de Madrid (CSV, versión posterior a septiembre de 2026).

**Objetivo:** quedarnos con las residencias, los centros de día y los centros de mayores de Madrid capital, con coordenadas, plazas y titularidad revisadas.

## 1. Carga y revisión general

In [1]:
import pandas as pd
import numpy as np

In [2]:
centros_raw = pd.read_csv("../raw/dato2y3/atencion_social_registro_centros.csv", sep=";", encoding="utf-8")
centros = centros_raw.copy()
centros.head()

,numero_registro,sector,tipo,subtipo,denominacion,plazas_autorizadas_numero,fecha_acreditacion,fecha_renovacion_acreditacion,municipio_codigo,municipio_nombre,...,direccion_demarcacion,direccion_zonass,direccion_coor_x,direccion_coor_y,contacto_telefono,contacto_correo,entidad_denominacion,entidad_nif,entidad_naturaleza,tipo_gestion
0,C0003,Personas con enfermedad mental crónica,CENTROS DE REHABILITACIÓN,CENTROS DE REHABILITACIÓN: OTROS,CENTRO DE REHABILITACION LABORAL NUEVA VIDA,40.0,NaN,NaN,079,Madrid,...,Salamanca 1,Guindalera,443644,4476188,91355 26 80,crl@crlnuevavida.es;pcarreras@crlnuevavida.es,ASOCIACION PSIQUIATRIA Y VIDA,G28727261,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS
1,C0005,Personas mayores,CENTROS RESIDENCIALES,RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,"NTRA. SRA. DE FATIMA, RESIDENCIA DE PERSONAS M...",96.0,27-05-2015,17-11-2026,061,Galapagar,...,San Lorenzo de El Escorial,Galapagar,415078,4491976,91 858 42 40,fatima@residencia-fatima.com;marina.hermoso@as...,"RESIDENCIA FATIMA, S.L.",B79025193,PRIVADA CON ÁNIMO DE LUCRO,PRIVADA + PEVS
2,C0007,Personas mayores,CENTROS RESIDENCIALES,RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,"SAN CAMILO, RESIDENCIA DE PERSONAS MAYORES",144.0,20-08-2026,20-08-2031,903,Tres Cantos,...,Tres Cantos,Tres Cantos I,439602,4496392,91 803 40 00,personas@sancamilo.org;residencia@sancamilo.or...,RR. CAMILOS-RESIDENCIA ASISTIDA SAN CAMILO,R2800714D,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS
3,C0010,Personas mayores,CENTROS DE DÍA,CENTROS DE DÍA: OTROS,"NUEVA EDAD, CENTRO DE DIA",60.0,20-05-2025,20-05-2030,079,Madrid,...,Ciudad Lineal 3,San Pascual,444737,4476780,914 033 131/902 095 902,asispa@asispa.org;vheranz@asispa.org,ASOCIACION SERVICIO INTEGRAL SECTORIAL PARA AN...,G28702504,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS
4,C0018,Personas mayores,CENTROS RESIDENCIALES,RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,"MADRE MARAVILLAS, RESIDENCIA DE PERSONAS MAYORES",54.0,20-08-2025,20-08-2030,065,Getafe,...,Zona Sur Rural,Perales del Río,444260,4463944,916847085-659381608,info@madremaravillas.com,OPTIMA CONSULTING 2024 S.L,B19352210,PRIVADA CON ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS


In [3]:
centros.info()
print(centros.shape)

<class 'pandas.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   numero_registro                2636 non-null   str    
 1   sector                         2636 non-null   str    
 2   tipo                           2636 non-null   str    
 3   subtipo                        2634 non-null   str    
 4   denominacion                   2636 non-null   str    
 5   plazas_autorizadas_numero      2632 non-null   float64
 6   fecha_acreditacion             1370 non-null   str    
 7   fecha_renovacion_acreditacion  1370 non-null   str    
 8   municipio_codigo               2636 non-null   str    
 9   municipio_nombre               2636 non-null   str    
 10  direccion_via_tipo             2636 non-null   str    
 11  direccion_via_codigo           2630 non-null   str    
 12  direccion_via_nombre           2636 non-null   str    
 13 

In [4]:
(centros.isnull().sum() / len(centros) * 100).round(2).sort_values(ascending=False)

direccion_mancomunidad           86.08
fecha_acreditacion               48.03
fecha_renovacion_acreditacion    48.03
contacto_correo                   5.35
direccion_coor_y                  3.22
direccion_coor_x                  3.22
entidad_nif                       0.76
contacto_telefono                 0.68
direccion_via_codigo              0.23
plazas_autorizadas_numero         0.15
subtipo                           0.08
direccion_demarcacion             0.08
direccion_zonass                  0.08
direccion_distrito                0.04
direccion_area                    0.04
municipio_codigo                  0.00
sector                            0.00
numero_registro                   0.00
tipo                              0.00
denominacion                      0.00
direccion_via_tipo                0.00
municipio_nombre                  0.00
direccion_codigo_postal           0.00
direccion_via_nombre              0.00
direccion_portal                  0.00
entidad_denominacion     

**Observación:** solo faltan datos relevantes en las coordenadas (3,2 %); direccion_mancomunidad (86 %) y las fechas de acreditación (48 %) no se usan en el proyecto.

In [5]:
centros["sector"].value_counts(dropna=False)

sector
Personas mayores                                        1072
Personas con discapacidad intelectual                    382
Personas con enfermedad mental crónica                   226
Otros grupos en situación de necesidad                   156
Personas sin hogar                                       153
Refugiados y asilados                                    137
Infancia y Adolescencia                                  134
Mujer                                                     77
Familia                                                   70
Infancia                                                  66
Adolescencia                                              63
Personas con discapacidad física                          57
(Inmigracion) Otros grupos en situación de necesidad      26
Presos y ex reclusos                                      12
Conjunto de la población                                   4
Colectivos en situación de necesidad                       1
Name: count, dtyp

**Observación:** 1.072 centros son de personas mayores, el sector que nos interesa; el resto (discapacidad, infancia, sin hogar...) queda fuera del estudio.

In [6]:
centros["tipo"].value_counts(dropna=False)

tipo
CENTROS RESIDENCIALES                       1565
CENTROS DE DÍA                               724
CENTROS DE REHABILITACIÓN                    205
CENTROS OCUPACIONALES                         93
CENTROS DE ACOGIDA                            48
CENTROS EXPERIMENTALES O NO CLASIFICADOS       1
Name: count, dtype: int64

**Observación:** nos quedaremos con centros residenciales y centros de día, que incluyen las residencias, los centros de día y los hogares y clubes.

In [7]:
centros["municipio_nombre"].value_counts(dropna=False).head(10)

municipio_nombre
Madrid                        1275
RESERVADO                      172
Leganés                         88
Móstoles                        57
Getafe                          56
Alcalá de Henares               56
Pozuelo de Alarcón              50
San Sebastián de los Reyes      45
Fuenlabrada                     41
Alcobendas                      38
Name: count, dtype: int64

**Observación:** Madrid capital tiene 1.275 centros; los 172 RESERVADO son centros protegidos (sobre todo de mujer e infancia) y quedan fuera al filtrar por Madrid y mayores.

## 2. Filtrado

- nos quedamos con los centros de Madrid capital, de personas mayores, residenciales o de día

In [8]:
centros = centros[
    (centros["municipio_nombre"] == "Madrid") &
    (centros["sector"] == "Personas mayores") &
    (centros["tipo"].isin(["CENTROS RESIDENCIALES", "CENTROS DE DÍA"]))
]
print(centros.shape)

(450, 28)


In [9]:
centros["subtipo"].value_counts(dropna=False)

subtipo
CENTROS DE DÍA: OTROS                           158
HOGARES Y CLUBES                                116
RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES    105
RESIDENCIAS DE PERSONAS MAYORES MIXTA            59
CENTROS RESIDENCIALES: OTROS                      5
RESIDENCIAS DE PERSONAS MAYORES AUTONOMAS         4
PISOS TUTELADOS                                   3
Name: count, dtype: int64

**Observación:**quedan 450 centros de mayores en Madrid capital; 168 residencias, 158 centros de día, 116 hogares y clubes y 8 de otros tipos residenciales.

**Decisión:** se mantienen los 8 pisos tutelados y apartamentos para mayores (subtipos "PISOS TUTELADOS" y "CENTROS RESIDENCIALES: OTROS"), porque también acogen a personas mayores expuestas al calor.

In [10]:
centros["numero_registro"].duplicated().sum()

np.int64(0)

In [11]:
centros["direccion_coor_x"] = pd.to_numeric(centros["direccion_coor_x"], errors="coerce")
centros["direccion_coor_y"] = pd.to_numeric(centros["direccion_coor_y"], errors="coerce")
print("Sin coordenadas:", centros["direccion_coor_x"].isnull().sum())

Sin coordenadas: 11


**Decisión:** se eliminan los 11 centros sin coordenadas (3 residencias, 4 centros de día y 4 hogares y clubes, unas 1.900 plazas). Queda como limitación del estudio.

In [12]:
centros = centros.dropna(subset=["direccion_coor_x", "direccion_coor_y"])
print(centros.shape)

(439, 28)


## 3. Selección de columnas

- nos quedamos con las columnas útiles para el proyecto y acortamos sus nombres

In [13]:
columnas_utiles = {
    "numero_registro": "codigo",
    "denominacion": "nombre",
    "subtipo": "subtipo",
    "plazas_autorizadas_numero": "plazas",
    "direccion_distrito": "distrito",
    "direccion_coor_x": "x",
    "direccion_coor_y": "y",
    "entidad_denominacion": "entidad",
    "entidad_naturaleza": "titularidad",
    "tipo_gestion": "tipo_gestion"
}
centros = centros[list(columnas_utiles.keys())].rename(columns=columnas_utiles)
centros.head()

,codigo,nombre,subtipo,plazas,distrito,x,y,entidad,titularidad,tipo_gestion
3,C0010,"NUEVA EDAD, CENTRO DE DIA",CENTROS DE DÍA: OTROS,60.0,Ciudad Lineal,444737.0,4476780.0,ASOCIACION SERVICIO INTEGRAL SECTORIAL PARA AN...,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA CON PLAZAS CONCERTADAS + PEVS
6,C0026,EL ENCUENTRO,HOGARES Y CLUBES,40.0,Ciudad Lineal,444238.0,4475696.0,ASOCIACION CLUB EL ENCUENTRO DE TERCERA EDAD,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA TOTAL
19,C0074,"SANTA FILIPINA, RESIDENCIA DE PERSONAS MAYORES",RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,38.0,Chamartín,442688.0,4480150.0,FUNDACION ASCE (ANTIGUAS ALUMNAS DEL SAGRADO C...,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA + PEVS
21,C0087,"RSDCIA. SACERDOTAL SAN PEDRO, RESIDENCIA DE PE...",RESIDENCIAS DE PERSONAS MAYORES MIXTA,95.0,Chamberí,440154.0,4475831.0,ASOCIACION CONGREGACION DE SAN PEDRO APOSTOL D...,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA + PEVS
31,C0138,"NTRA. SRA. DEL PILAR, RESIDENCIA DE PERSONAS M...",RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES,55.0,San Blas,448266.0,4477240.0,CASA COMUNIDAD HERMANAS MERCEDARIAS DE LA CARIDAD,PRIVADA SIN ÁNIMO DE LUCRO,PRIVADA + PEVS


- revisamos nulos y valores de las plazas

In [14]:
print("Plazas nulas:", centros["plazas"].isnull().sum())
centros["plazas"].describe()

Plazas nulas: 0


count     439.000000
mean      138.157175
std       136.868201
min         7.000000
25%        45.000000
50%        94.000000
75%       176.500000
max      1000.000000
Name: plazas, dtype: float64

In [15]:
centros["titularidad"].value_counts(dropna=False)

titularidad
PÚBLICA                       206
PRIVADA CON ÁNIMO DE LUCRO    182
PRIVADA SIN ÁNIMO DE LUCRO     51
Name: count, dtype: int64

In [16]:
centros["tipo_gestion"].value_counts(dropna=False)

tipo_gestion
PRIVADA TOTAL                            175
PRIVADA + PEVS                           123
PRIVADA CON PLAZAS CONCERTADAS + PEVS     64
PROPIA DE GESTIÓN DIRECTA                 37
PROPIA DE GESTIÓN INDIRECTA               28
PRIVADA CON PLAZAS CONCERTADAS            12
Name: count, dtype: int64

## 4. Tipo de centro

- creamos una columna con el tipo de centro simplificado a partir del subtipo

In [17]:
tipos = {
    "RESIDENCIAS DE PERSONAS MAYORES DEPENDIENTES": "residencia",
    "RESIDENCIAS DE PERSONAS MAYORES MIXTA": "residencia",
    "RESIDENCIAS DE PERSONAS MAYORES AUTONOMAS": "residencia",
    "CENTROS DE DÍA: OTROS": "centro de día",
    "HOGARES Y CLUBES": "centro de mayores",
    "PISOS TUTELADOS": "vivienda tutelada",
    "CENTROS RESIDENCIALES: OTROS": "vivienda tutelada"
}
centros["tipo_centro"] = centros["subtipo"].map(tipos)
centros["tipo_centro"].value_counts(dropna=False)

tipo_centro
residencia           165
centro de día        154
centro de mayores    112
vivienda tutelada      8
Name: count, dtype: int64

Observación: todos los subtipos quedan clasificados; las residencias y los centros de día son los grupos más grandes.

In [18]:
centros.groupby("tipo_centro")["plazas"].describe()

,count,mean,std,min,25%,50%,75%,max
tipo_centro,,,,,,,,
centro de día,154.0,60.694805,36.238016,7.0,40.00,45.0,75.0,235.0
centro de mayores,112.0,272.348214,178.580474,38.0,148.50,226.5,359.0,1000.0
residencia,165.0,122.418182,92.633231,11.0,55.00,97.0,170.0,601.0
vivienda tutelada,8.0,75.250000,78.846596,16.0,25.25,38.5,104.5,246.0


Observación: los centros de mayores tienen una mediana de 226 plazas y llegan a 1.000, frente a 97 en residencias y 45 en centros de día; confirma que sus “plazas” son aforo o socios, no plazas de atención.

**Nota:** las plazas de los centros de mayores reflejan aforo o socios, no plazas de atención; no son comparables con las de residencias y centros de día al calcular la exposición del índice de riesgo.